# 01 Data Setup And Train GAN

Use this notebook for dataset sanity checks, image previews, and a standalone GAN run in `runs/gan/`. On shared CUDA hosts, keep `DEVICE_OVERRIDE="auto"` unless you want to pin a specific GPU. If a pinned GPU is nearly full, switch back to `auto` or another GPU and restart the kernel before rerunning.


In [ ]:
from pathlib import Path
import os
import sys
from pprint import pprint

from IPython.display import display
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from workflow import sample_image_paths, summarize_dataset, train_gan

ROOT


In [ ]:
# Edit these values before running.
cfg = Config().with_runtime_profile_defaults()
DATA_ROOT = ROOT / "data_final"
OUT_DIR = ROOT / "runs" / "gan"
TRAIN_N_PER_CLASS = None
STRICT_FID = True

# Shared-GPU overrides. Leave DEVICE_OVERRIDE="auto" to use the visible CUDA
# device with the most free memory. If a pinned GPU is nearly full, switch
# back to "auto" or another GPU and restart the kernel before rerunning.
DEVICE_OVERRIDE = "auto"  # e.g. "cuda:0" or "cuda:1"
NUM_WORKERS_OVERRIDE = None

# Optional quick smoke-run overrides for notebook debugging.
SMOKE_GAN_OVERRIDES = {}
overrides = dict(SMOKE_GAN_OVERRIDES)
if DEVICE_OVERRIDE != "auto":
    overrides["device"] = DEVICE_OVERRIDE
if NUM_WORKERS_OVERRIDE is not None:
    overrides["num_workers"] = NUM_WORKERS_OVERRIDE
if overrides:
    cfg = cfg.with_overrides(**overrides)

{
    "cfg": cfg,
    "resolved_device": cfg.resolve_device(),
    "loader_options": cfg.loader_options(),
    "data_root": DATA_ROOT,
    "out_dir": OUT_DIR,
    "train_n_per_class": TRAIN_N_PER_CLASS,
    "strict_fid": STRICT_FID,
}


In [ ]:
dataset_summary = summarize_dataset(DATA_ROOT)
pprint(dataset_summary)

for class_name, sample_path in sample_image_paths(DATA_ROOT):
    print(class_name, sample_path.name)
    display(Image.open(sample_path))


## GAN Training

This notebook keeps the standalone GAN workflow available. For the full Task 1 pipeline, use `02_generate_synthetic_data_and_classifier_experiments.ipynb`.


In [ ]:
gan_run = train_gan(
    cfg,
    data_root=DATA_ROOT,
    fid_root=DATA_ROOT,
    out_dir=OUT_DIR,
    train_n_per_class=TRAIN_N_PER_CLASS,
    strict_fid=STRICT_FID,
)
gan_run["summary"]
